In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/content/GWSD.csv")

In [ ]:
df.shape

(2300, 18)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2300 entries, 0 to 2299
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        2300 non-null   int64  
 1   round             2300 non-null   int64  
 2   batch             2300 non-null   int64  
 3   sent_id           2300 non-null   object 
 4   sentence          2300 non-null   object 
 5   worker_0          2300 non-null   object 
 6   worker_1          2300 non-null   object 
 7   worker_2          2300 non-null   object 
 8   worker_3          2300 non-null   object 
 9   worker_4          2300 non-null   object 
 10  worker_5          2300 non-null   object 
 11  worker_6          2300 non-null   object 
 12  worker_7          2300 non-null   object 
 13  disagree          2050 non-null   float64
 14  agree             2050 non-null   float64
 15  neutral           2050 non-null   float64
 16  guid              2300 non-null   object 


In [ ]:
df = df.dropna(subset=["agree", "disagree", "neutral"])

In [ ]:
df["label"] = (
    df[["agree", "disagree", "neutral"]]
    .idxmax(axis=1)
    .replace({
        "agree": "FAVOR",
        "disagree": "AGAINST",
        "neutral": "NONE"
    })
)

In [ ]:
final_df = df[["sentence", "label"]]

In [ ]:
final_df.head()

,sentence,label
5,Warmer-than-normal sea surface temperatures ar...,FAVOR
6,We will continue to rely in part on fossil fue...,NONE
7,The actual rise in sea levels measured only 1....,NONE
8,Claims of global warming have been greatly exa...,AGAINST
9,The Intergovernmental Panel on Climate Change ...,NONE


In [ ]:
final_df.shape

(2050, 2)

In [ ]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2050 entries, 5 to 2299
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   sentence  2050 non-null   object
 1   label     2050 non-null   object
dtypes: object(2)
memory usage: 48.0+ KB


In [ ]:
final_df['label'].value_counts()

,count
label,
NONE,873
FAVOR,777
AGAINST,400


In [ ]:
final_df.to_csv("final_GWSD.csv", index=False)

In [ ]:
import pandas as pd
import re
import string

# --------------------------------------------------
# General Text Cleaning Function (Language Independent)
# --------------------------------------------------

def clean_text(text, remove_punctuation=True):
    """
    General text preprocessing function.

    Steps:
    1. Convert to string
    2. Remove URLs
    3. Remove HTML tags
    4. Remove @mentions
    5. Remove hashtags symbol (#) but keep the word
    6. Remove emojis
    7. Remove punctuation (optional)
    8. Normalize whitespace
    """

    # Handle missing values
    if pd.isna(text):
        return ""

    text = str(text)

    # -------------------------
    # Remove URLs
    # -------------------------
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # -------------------------
    # Remove HTML Tags
    # -------------------------
    text = re.sub(r'<.*?>', '', text)

    # -------------------------
    # Remove Mentions
    # -------------------------
    text = re.sub(r'@\w+', '', text)

    # -------------------------
    # Remove '#' only
    # Example:
    # #Climate -> Climate
    # -------------------------
    text = re.sub(r'#(\w+)', r'\1', text)

    # -------------------------
    # Remove Emojis
    # -------------------------
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F700-\U0001F77F"
        u"\U0001F780-\U0001F7FF"
        u"\U0001F800-\U0001F8FF"
        u"\U0001F900-\U0001F9FF"
        u"\U0001FA00-\U0001FAFF"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE,
    )
    text = emoji_pattern.sub("", text)

    # -------------------------
    # Remove punctuation (optional)
    # -------------------------
    if remove_punctuation:
        text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)

    # -------------------------
    # Normalize whitespace
    # -------------------------
    text = re.sub(r"\s+", " ", text).strip()

    return text


# --------------------------------------------------
# Dataset Preprocessing Function
# --------------------------------------------------

def preprocess_dataset(df, text_column, remove_punctuation=True):
    """
    Preprocess an entire dataset.

    Parameters
    ----------
    df : pandas DataFrame
    text_column : column containing text
    remove_punctuation : True/False

    Returns
    -------
    Cleaned DataFrame
    """

    # Remove null values
    df = df.dropna(subset=[text_column])

    # Remove duplicate rows
    df = df.drop_duplicates()

    # Clean text
    df[text_column] = df[text_column].apply(
        lambda x: clean_text(x, remove_punctuation)
    )

    # Remove empty rows after cleaning
    df = df[df[text_column].str.strip() != ""]

    # Reset index
    df = df.reset_index(drop=True)

    return df

In [ ]:
# Load dataset
df = pd.read_csv("/content/final_GWSD.csv")

In [ ]:
# Preprocess
clean_df = preprocess_dataset(
    df,
    text_column="sentence",
    remove_punctuation=True
)

# Save cleaned dataset
clean_df.to_csv("cleaned_dataset.csv", index=False)

print(clean_df.head())

                                            sentence    label
0  Warmerthannormal sea surface temperatures are ...    FAVOR
1  We will continue to rely in part on fossil fue...     NONE
2  The actual rise in sea levels measured only 12...     NONE
3  Claims of global warming have been greatly exa...  AGAINST
4  The Intergovernmental Panel on Climate Change ...     NONE


In [ ]:
clean_df.shape

(2043, 2)

In [ ]:
clean_df['label'].value_counts()

,count
label,
NONE,872
FAVOR,774
AGAINST,397


In [ ]:
#end gwsd

In [ ]:
df=pd.read_csv("/content/semeval2016_climate data.csv")

In [ ]:
df.shape

(395, 5)

In [ ]:
df.head()

,Tweet,Target,Stance,Opinion Towards,Sentiment
0,"We cant deny it, its really happening. #SemST",Climate Change is a Real Concern,FAVOR,1. The tweet explicitly expresses opinion abo...,other
1,RT @cderworiz: Timelines are short. Strategy m...,Climate Change is a Real Concern,FAVOR,1. The tweet explicitly expresses opinion abo...,pos
2,SO EXCITING! Meaningful climate change action ...,Climate Change is a Real Concern,FAVOR,1. The tweet explicitly expresses opinion abo...,pos
3,"Delivering good jobs for Albertans, maintainin...",Climate Change is a Real Concern,FAVOR,1. The tweet explicitly expresses opinion abo...,pos
4,@davidswann says he wants carbon fund to be sp...,Climate Change is a Real Concern,FAVOR,3. The tweet is not explicitly expressing opi...,other


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Tweet            395 non-null    object
 1   Target           395 non-null    object
 2   Stance           395 non-null    object
 3   Opinion Towards  395 non-null    object
 4   Sentiment        395 non-null    object
dtypes: object(5)
memory usage: 15.6+ KB


In [ ]:
df = df[["Tweet", "Stance"]]

In [ ]:
df = df.rename(columns={
    "Tweet": "sentence",
    "Stance": "label"
})

In [ ]:
df.head()

,sentence,label
0,"We cant deny it, its really happening. #SemST",FAVOR
1,RT @cderworiz: Timelines are short. Strategy m...,FAVOR
2,SO EXCITING! Meaningful climate change action ...,FAVOR
3,"Delivering good jobs for Albertans, maintainin...",FAVOR
4,@davidswann says he wants carbon fund to be sp...,FAVOR


In [ ]:
df["label"].value_counts()

,count
label,
FAVOR,212
NONE,168
AGAINST,15


In [ ]:
df.to_csv("sentence_label_dataset.csv", index=False)

In [ ]:
df.shape

(395, 2)

In [ ]:
import pandas as pd

df = pd.read_csv("/content/sentence_label_dataset.csv")

clean_df = preprocess_dataset(
    df,
    text_column="sentence",
    remove_punctuation=True
)

clean_df.to_csv("clean_final_semeval2016-task6-trainingdata_dataset.csv", index=False)

print(clean_df.head())

                                            sentence  label
0         We cant deny it its really happening SemST  FAVOR
1  RT Timelines are short Strategy must be in pla...  FAVOR
2  SO EXCITING Meaningful climate change action i...  FAVOR
3  Delivering good jobs for Albertans maintaining...  FAVOR
4  says he wants carbon fund to be spent on publi...  FAVOR


In [ ]:
clean_df.shape

(395, 2)

In [ ]:
df["label"].value_counts()

,count
label,
FAVOR,212
NONE,168
AGAINST,15


In [ ]:
# Load the first cleaned dataset (from GWSD)
gwsd_clean_df = pd.read_csv('/content/cleaned_dataset.csv')

# Load the second cleaned dataset (from Semeval)
semeval_clean_df = pd.read_csv('/content/clean_final_semeval2016-task6-trainingdata_dataset.csv')

# Concatenate the two dataframes
merged_df = pd.concat([gwsd_clean_df, semeval_clean_df], ignore_index=True)

print(f"Shape of merged DataFrame: {merged_df.shape}")
print("First 5 rows of merged DataFrame:")
display(merged_df.head())

Shape of merged DataFrame: (2438, 2)
First 5 rows of merged DataFrame:


,sentence,label
0,Warmerthannormal sea surface temperatures are ...,FAVOR
1,We will continue to rely in part on fossil fue...,NONE
2,The actual rise in sea levels measured only 12...,NONE
3,Claims of global warming have been greatly exa...,AGAINST
4,The Intergovernmental Panel on Climate Change ...,NONE


In [ ]:
merged_df.shape

(2438, 2)

In [ ]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2438 entries, 0 to 2437
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   sentence  2438 non-null   object
 1   label     2438 non-null   object
dtypes: object(2)
memory usage: 38.2+ KB


In [ ]:
merged_df.to_csv("final merge_dataset.csv", index=False)